# MCP: safecmd

> MCP server exposing safecmd command validation and safe execution as Claude Code tools

In [ ]:
#| default_exp mcp_safecmd

In [ ]:
#| export
from __future__ import annotations
import json
import subprocess
import sys
from pathlib import Path

try:
    from mcp.server import Server
    from mcp.server.stdio import stdio_server
    from mcp import types
    HAS_MCP = True
except ImportError:
    HAS_MCP = False

In [ ]:
#| export
def _get_allowlist() -> list[str]:
    """Load allowlist from .claude/safecmd_allowlist.json or return defaults."""
    from claude_plugins.hooks.safecmd_hook import load_allowlist, DEFAULT_ALLOWLIST
    try:
        return load_allowlist()
    except Exception:
        return DEFAULT_ALLOWLIST

In [ ]:
#| export
def validate_command_tool(command: str) -> dict:
    """Validate a shell command against the safecmd allowlist."""
    from claude_plugins.hooks.safecmd_hook import validate_command
    allowlist = _get_allowlist()
    allowed, reason = validate_command(command, allowlist)
    return {
        'allowed': allowed,
        'reason': reason,
        'command': command,
        'allowlist_size': len(allowlist),
    }

In [ ]:
#| export
def run_safe_tool(command: str, timeout: int = 60) -> dict:
    """Validate and run a shell command safely. Only runs if allowlist check passes."""
    validation = validate_command_tool(command)
    if not validation['allowed']:
        return {
            'success': False,
            'error': f'Command blocked: {validation["reason"]}',
            'command': command,
        }
    try:
        result = subprocess.run(
            command, shell=True, capture_output=True, text=True, timeout=timeout
        )
        return {
            'success': result.returncode == 0,
            'returncode': result.returncode,
            'stdout': result.stdout,
            'stderr': result.stderr,
            'command': command,
        }
    except subprocess.TimeoutExpired:
        return {'success': False, 'error': f'Command timed out after {timeout}s', 'command': command}
    except Exception as e:
        return {'success': False, 'error': str(e), 'command': command}

In [ ]:
#| export
def create_server() -> 'Server':
    server = Server('safecmd')

    @server.list_tools()
    async def list_tools() -> list[types.Tool]:
        return [
            types.Tool(
                name='validate_command',
                description='Check if a shell command is in the safecmd allowlist before running it.',
                inputSchema={
                    'type': 'object',
                    'properties': {
                        'command': {'type': 'string', 'description': 'The shell command to validate'},
                    },
                    'required': ['command'],
                },
            ),
            types.Tool(
                name='run_safe',
                description='Validate and execute a shell command safely. Blocked if not in allowlist.',
                inputSchema={
                    'type': 'object',
                    'properties': {
                        'command': {'type': 'string', 'description': 'The shell command to run'},
                        'timeout': {'type': 'integer', 'description': 'Timeout in seconds', 'default': 60},
                    },
                    'required': ['command'],
                },
            ),
        ]

    @server.call_tool()
    async def call_tool(name: str, arguments: dict) -> list[types.TextContent]:
        if name == 'validate_command':
            result = validate_command_tool(arguments['command'])
        elif name == 'run_safe':
            result = run_safe_tool(arguments['command'], timeout=arguments.get('timeout', 60))
        else:
            result = {'error': f'Unknown tool: {name}'}
        return [types.TextContent(type='text', text=json.dumps(result, indent=2))]

    return server

In [ ]:
#| export
def main():
    if not HAS_MCP:
        print('mcp package not installed. Run: uv add mcp', file=sys.stderr)
        sys.exit(1)
    import asyncio
    server = create_server()
    asyncio.run(stdio_server(server))


if __name__ == '__main__':
    main()